# Apply Simple Hill Climbing to a Maze Problem

This notebook demonstrates how hill climbing local search behaves applies to the L-Maze problem.

## Setup

In [18]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("L_maze.txt", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Artificial_Intelligence/refs/heads/master/Local_Search/")
download("maze_helper.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Artificial_Intelligence/refs/heads/master/Local_Search/")

## Search Helper

We need to define an expand function that will produce the neighbors for local search.

In [19]:
DIR2POS = {
    'N' : (-1, 0),
    'E' : ( 0, 1),
    'S' : ( 1, 0),
    'W' : ( 0,-1)
}

ORDER = None
RANDOMIZATION = None
DIRS = None 

# The order matters if RANDOMIZATION = False
# Note that: DFS works takes elements from the frontier in reverse order!
def set_order(order = "NESW", random = False, verbose = True):
    global RANDOMIZATION, DIR2POS, ORDER, DIRS
    
    RANDOMIZATION = random
    ORDER = [pos for pos in order]
    DIRS = [DIR2POS[dir] for dir in ORDER]
    
    if verbose:
        if RANDOMIZATION:
            print("Directions are checked at every step in random order.")
        else:
            print(f"Directions are checked in the order {ORDER}")
        
set_order("NESW", verbose = True)

Directions are checked in the order ['N', 'E', 'S', 'W']


In [20]:
def expand(maze, pos):
    """find available directions as a list of lists with elements [dir, (pos)]."""
    global RANDOMIZATION, DIRS, ORDER
       
    available = [[dir, pos] for dir, pos in zip(ORDER, np.add(pos, DIRS)) if mh.look(maze, pos) != "X"]
    if RANDOMIZATION: np.random.shuffle(available)
        
    return available

# Hill Climbing

We use Manhattan distance to the goal as the objective function (minimization).

In [22]:
import numpy as np

# Manhattan distance h(n)
def manhattan(pos1, pos2):
    """returns the Manhattan distance between two positions"""
    return(np.sum(np.abs(np.subtract(pos1, pos2))))
    
print(manhattan([0, 0], [1, 1]))

2


In [23]:
def local_search(maze, vis = False, animation = False, verbose = False):
    if animation:
        maze_anim = []
    else:
        maze_anim = None
    
    # Local search starts here
    start_pos = mh.find_pos(maze, "S")
    goal_pos = mh.find_pos(maze, "G")

    current = start_pos
    current_obj = manhattan(current, goal_pos)

    while True:
        # find best neighbor
        neighbors = [n[1] for n in expand(maze, current)]
        obj = [manhattan(n, goal_pos) for n in neighbors]
        best_obj = min(obj)
        best_neighbor = neighbors[obj.index(best_obj)]

        if verbose:
            print("Current state:", current, "with obj", current_obj)
            print("Evaluating neighbors:\t", neighbors)
            print("Obj:\t\t\t", obj)


        # move there if it is an improvement or stop
        if best_obj < current_obj:
            current = best_neighbor
            current_obj = best_obj
            if verbose:
                print("Moving to:", current, "\n")
        else:
            if verbose:
                print("No better state in the neighborhood! Stopping.\n")
            break

        if animation:
            maze2 = maze.copy()
            maze2[current[0], current[1]] = "P"
            maze_anim.append(maze2)


    return { "solution" : current, "path" : None, "reached" : None, "maze_anim" : maze_anim }

## Run the Algorithm

In [27]:
import numpy as np
import maze_helper as mh
import os

# make sure the files are in the current working directory
# Make sure the maze file is in the current working directory and
# provide a compatibility loader if the downloaded helper is older.

if not os.path.exists("L_maze.txt"):
    raise FileNotFoundError("L_maze.txt not found in the working directory. Run the download cell first.")

# Some versions of maze_helper.py do not define load_maze.
if not hasattr(mh, "load_maze"):
    def load_maze(filename):
        with open(filename, "r", encoding="utf-8") as f:
            rows = [list(line.rstrip("\n")) for line in f if line.strip()]
        return np.array(rows, dtype="<U1")

    mh.load_maze = load_maze

maze = mh.load_maze("L_maze.txt")


**Note:** Simple hill climbing is greedy. It gets stuck in local optima!

In [29]:
# Bài 1: Thử thay đổi thứ tự xét hướng (ORDER) và lựa chọn ngẫu nhiên (RANDOMIZATION)
goal_pos = mh.find_pos(maze, "G")
start_pos = mh.find_pos(maze, "S")
print(f"Vị trí xuất phát S: {start_pos}, Vị trí đích G: {goal_pos}")
print(f"Khoảng cách Manhattan ban đầu h(S): {manhattan(start_pos, goal_pos)}\n")

orders_to_test = ["NESW", "ESWN", "SWNE", "WNES"]
for ord_str in orders_to_test:
    set_order(order=ord_str, random=False, verbose=False)
    res = local_search(maze, animation=False, verbose=False)
    sol = tuple(res["solution"])
    print(f"Thứ tự cố định {ord_str} -> Dừng tại: {sol}, h(n) = {manhattan(sol, goal_pos)}, Đạt đích G? {sol == goal_pos}")

print("\nChạy 5 lần với RANDOMIZATION = True:")
np.random.seed(42)
for i in range(1, 6):
    set_order(order="NESW", random=True, verbose=False)
    res = local_search(maze, animation=False, verbose=False)
    sol = tuple(res["solution"])
    print(f"Lần chạy {i} (Random) -> Dừng tại: {sol}, h(n) = {manhattan(sol, goal_pos)}, Đạt đích G? {sol == goal_pos}")

Vị trí xuất phát S: (np.int64(9), np.int64(5)), Vị trí đích G: (np.int64(2), np.int64(12))
Khoảng cách Manhattan ban đầu h(S): 14

Thứ tự cố định NESW -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False
Thứ tự cố định ESWN -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False
Thứ tự cố định SWNE -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False
Thứ tự cố định WNES -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False

Chạy 5 lần với RANDOMIZATION = True:
Lần chạy 1 (Random) -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False
Lần chạy 2 (Random) -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False
Lần chạy 3 (Random) -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False
Lần chạy 4 (Random) -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False
Lần chạy 5 (Random) -> Dừng tại: (np.int64(5), np.int64(9)), h(n) = 6, Đạt đích G? False


## Phân tích và Nhận xét Bài 1 (Simple Hill Climbing trên Mê cung)

1. **Hàm mục tiêu và sinh trạng thái lân cận:**
   * Hàm `expand(maze, pos)` kiểm tra 4 hướng `N (-1, 0)`, `E (0, 1)`, `S (1, 0)`, `W (0, -1)` xung quanh vị trí hiện tại và lọc bỏ các ô tường `'X'`.
   * Hàm mục tiêu là khoảng cách Manhattan đến đích $G(2, 12)$: $h(n) = \vert{}r_n - r_G\vert{} + \vert{}c_n - c_G\vert{}$. Bài toán tìm kiếm cực tiểu hóa $h(n)$ (bước đi hợp lệ được chấp nhận khi $h(\text{neighbor}) < h(\text{current})$).

2. **Kết quả `result["solution"]` và giải thích nguyên nhân dừng sớm:**
   * Xuất phát từ $S(9, 5)$ với $h(S) = \vert{}9-2\vert{} + \vert{}5-12\vert{} = 14$, thuật toán đi lên phía Bắc đến $(5, 5)$ rồi rẽ sang Đông tới vị trí dừng `result["solution"] = [5, 9]` với $h([5, 9]) = \vert{}5-2\vert{} + \vert{}9-12\vert{} = 6$.
   * Vị trí dừng $(5, 9)$ **chưa đến được đích** $G(2, 12)$ ($h(G) = 0$).
   * **Nguyên nhân:** Tại $(5, 9)$, bức tường chữ L chặn cả hướng Bắc $(4, 9)$ lẫn hướng Đông $(5, 10)$. Hai ô lân cận hợp lệ duy nhất là đi lùi sang Tây $(5, 8)$ hoặc đi xuống Nam $(6, 9)$, cả hai đều có $h = 7 > 6$. Vì Simple Hill Climbing mang tính tham lam thuần túy (chỉ chấp nhận bước giảm $h(n)$), thuật toán bị **mắc kẹt tại cực trị cục bộ (local minimum)** ở góc tường chữ L.

3. **Ảnh hưởng của `ORDER` và `RANDOMIZATION`:**
   * Khi đổi thứ tự ưu tiên hướng (`NESW`, `ESWN`, `SWNE`, `WNES`) hoặc bật `RANDOMIZATION = True`, đường đi cụ thể trong hành lang có thể thay đổi (ví dụ: ưu tiên đi sang Đông trước rồi mới đi lên Bắc), nhưng **tất cả các lần chạy đều bị kẹt tại cùng điểm cực trị cục bộ $(5, 9)$** với $h = 6$. Để thoát khỏi bẫy hình chữ L này, thuật toán bắt buộc phải chấp nhận các bước đi tạm thời làm tăng $h(n)$ (như Simulated Annealing) hoặc lưu trữ không gian trạng thái đã duyệt (như BFS/A*).